# Solutions 09: Student Initialisation, Prune-Then-Distill

This notebook solves the four exercises of Lab 09. Execution status: all four solutions run
live on the 135M model; the parts of exercises 1, 2, and 4 that need the 1.7B teacher or a
training loop are written out but gated behind `RUN_TRAINING = False`, with expected results
stated from the lab's Part C. Attempt the exercises yourself before reading this file; the
value of an exercise is the attempt, and a solution read too early converts a measurement
into trivia.

One honest note about scale before anything runs. The exercises as written target the 1.7B
teacher. Everything here demonstrates the *method* on the 135M model instead, because the
method (measure importance, prune, probe, compare) is identical at every scale, and because
the 135M version executes on a laptop-class CPU in minutes. Where the scale change matters
for the conclusion, the interpretation cell says so explicitly.

In [1]:
import sys, os, json, math, copy
os.environ.setdefault("HF_HUB_OFFLINE", "1")          # everything needed is already cached
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
sys.path.insert(0, "../code")

import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from kd_core import masked_mean
from kd_pipeline import set_seed_everywhere, config_fingerprint

RUN_TRAINING = False        # <-- flip on the training box
SEED = 17
set_seed_everywhere(SEED)

from transformers import AutoModelForCausalLM
from transformers.utils import logging as hf_logging
hf_logging.disable_progress_bar()

PROBE_MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
model = AutoModelForCausalLM.from_pretrained(PROBE_MODEL, dtype=torch.float32).eval()
n_layers = model.config.num_hidden_layers

# Probe set: 8 rows, truncated to 192 tokens so every sweep cell stays inside a
# CPU minute-budget. The lab measured its importance profile on 8 x 384; the
# truncation changes absolute losses a little and the ranking's use not at all,
# because the ranking is only ever read as an ordering.
ev = torch.load("../data/lab03/eval.pt")
PROBE_T = 192
probe_ids, probe_mask = ev["input_ids"][:8, :PROBE_T], ev["mask"][:8, :PROBE_T]

@torch.no_grad()
def probe_loss(m, ids=None, mask=None):
    ids = probe_ids if ids is None else ids
    mask = probe_mask if mask is None else mask
    logits = m(ids).logits
    lp = F.log_softmax(logits[:, :-1].float(), dim=-1)
    nll = -lp.gather(-1, ids[:, 1:].unsqueeze(-1)).squeeze(-1)
    return float(masked_mean(nll, mask[:, 1:]))

def depth_prune(src_model, keep_layers):        # the lab's Part A-2 function, verbatim
    keep_layers = sorted(keep_layers)
    cfg = copy.deepcopy(src_model.config)
    cfg.num_hidden_layers = len(keep_layers)
    pruned = AutoModelForCausalLM.from_config(cfg)
    src, dst = src_model.state_dict(), {}
    for name, tensor in src.items():
        if ".layers." in name:
            lid = int(name.split(".layers.")[1].split(".")[0])
            if lid in keep_layers:
                new_lid = keep_layers.index(lid)
                dst[name.replace(f".layers.{lid}.", f".layers.{new_lid}.")] = tensor
        else:
            dst[name] = tensor
    missing, unexpected = pruned.load_state_dict(dst, strict=False)
    assert not missing and not unexpected, (missing, unexpected)
    return pruned

prof = json.load(open("../data/lab09_importance_135m.json"))
damage = prof["damage"]
ranked = sorted(range(n_layers), key=lambda i: damage[i])   # least important first
base = probe_loss(model)
assert len(damage) == n_layers, "saved profile must cover every layer"
assert int(probe_mask.sum()) > 100, "probe must contain real completion tokens"
print(f"{PROBE_MODEL}: {n_layers} layers | probe 8 x {PROBE_T} | base loss {base:.3f} nats")
print(f"saved importance profile loaded; least important layers: {ranked[:6]}")

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HuggingFaceTB/SmolLM2-135M-Instruct: 30 layers | probe 8 x 192 | base loss 0.908 nats
saved importance profile loaded; least important layers: [5, 4, 12, 3, 7, 9]


## Exercise 1: The capacity control

**The exercise, restated.** Prune the 1.7B teacher all the way down to a 360M-equivalent
model, distill it at the same budget, and complete the per-parameter comparison Part C left
open. The lab warns that depth-only pruning "gets ugly" below roughly 50% layer removal,
because Minitron thins width as well as depth when it cuts this deep, and this exercise is
where you observe that ugliness yourself.

**The approach.** The 1.7B surgery and the distillation runs need the training box, so they
are gated. What runs live is the exercise's actual premise: the shape of the degradation
curve as depth pruning cuts deeper. The 135M model has 30 layers; using the saved importance
profile from the lab's Part A-1, I prune it to 26, 22, 18, and 14 layers (removing 13%, 27%,
40%, and 53% of its depth) and measure the probe loss at each stop. The 14-layer point is
past the 50% mark where the ugliness is supposed to begin. If the exercise's premise is
right, the curve should not be a straight line: each additional slice of removed depth should
cost more nats than the one before it, because the surviving layers have progressively less
redundancy left to absorb the damage. The plot and two asserts check exactly that: monotone
degradation, and an accelerating (convex) tail.

In [2]:
KEEPS = [26, 22, 18, 14]
sweep = {n_layers: base}
pruned_models = {}
for n_keep in KEEPS:
    drop = set(ranked[:n_layers - n_keep])          # least important, per saved profile
    keep = [i for i in range(n_layers) if i not in drop]
    p = depth_prune(model, keep).eval()
    sweep[n_keep] = probe_loss(p)
    if n_keep == 22:
        keep22, loss22 = keep, sweep[22]            # reused by exercises 2, 3, 4
        model22 = p                                  # keep the 22-layer patient around
    else:
        del p
    removed = n_layers - n_keep
    print(f"keep {n_keep:>2} layers (remove {removed:>2}, {removed/n_layers:.0%}): "
          f"probe loss {sweep[n_keep]:.3f} nats (+{sweep[n_keep]-base:.3f})")

xs = [n_layers - k for k in sorted(sweep, reverse=True)]         # layers removed
ys = [sweep[k] for k in sorted(sweep, reverse=True)]
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(xs, ys, "o-", lw=2)
ax.axvline(n_layers * 0.5, ls="--", color="#c33", lw=1)
ax.text(n_layers * 0.5 + 0.3, ys[0], "50% removed", color="#c33", fontsize=9)
ax.set_xlabel("layers removed (of 30, least-important first)")
ax.set_ylabel("probe loss (nats)")
ax.set_title("depth-only pruning: where it gets ugly")
fig.tight_layout(); fig.savefig("../figures/sol09_depth_sweep.png", dpi=120)
print("curve written to ../figures/sol09_depth_sweep.png")

losses = [sweep[k] for k in [n_layers] + KEEPS]
assert all(a < b for a, b in zip(losses, losses[1:])), \
    "removing more layers must monotonically worsen the probe loss"
marg_first = sweep[26] - base                        # cost of the first 4 removed layers
marg_last = sweep[14] - sweep[18]                    # cost of the last 4 removed layers
assert marg_last > marg_first, \
    "the curve must be convex: the last slice of depth costs more than the first"
print(f"marginal cost, first 4 layers removed: +{marg_first:.3f} nats; "
      f"last 4 (crossing 50%): +{marg_last:.3f} nats -> degradation accelerates")

keep 26 layers (remove  4, 13%): probe loss 1.567 nats (+0.658)


keep 22 layers (remove  8, 27%): probe loss 2.757 nats (+1.849)


keep 18 layers (remove 12, 40%): probe loss 3.837 nats (+2.929)


keep 14 layers (remove 16, 53%): probe loss 5.328 nats (+4.420)
curve written to ../figures/sol09_depth_sweep.png
marginal cost, first 4 layers removed: +0.658 nats; last 4 (crossing 50%): +1.491 nats -> degradation accelerates


**Interpretation.** The printed sweep and the saved curve show the exercise's premise
holding on a real model. Both asserts passed: the loss rises monotonically as depth is
removed (0.91 intact, then 1.57, 2.76, 3.84, 5.33 nats at 26, 22, 18, 14 layers), and the
marginal cost accelerates: the first four removed layers cost +0.66 nats while the four that
cross the 50% line cost +1.49, more than twice as much per layer removed. That
acceleration is the "gets ugly" the exercise asks you to observe, and the mechanism is
redundancy exhaustion: the first layers removed are the ones the importance profile says the
model can spare, and every later slice must come from layers the model spares less and less.

What this means for the gated 1.7B version: cutting 24 layers down to the 5 or 6 that a
360M-equivalent depth would allow sits far beyond the curve's knee, so a depth-only 360M
patient would start distillation in far worse shape than the pretrained 360M checkpoint it is
supposed to challenge. The expected outcome of the gated run, grounded in Part C, is that
depth-only prune-to-360M *loses* the per-parameter comparison against the pretrained 360M,
and that closing the gap requires Minitron-style width pruning, which shrinks each layer's
hidden dimensions rather than deleting more layers. The honest completion of Part C's open
comparison is therefore: the 1.0B pruned model's win over the 360M checkpoint was partly
capacity, and depth-only pruning cannot cash that capacity in at matched size. The gated code
path is one `depth_prune` call plus one Lab 04 stage-2 run per arm, unchanged from the lab's
Part B.

## Exercise 2: Iterative vs one-shot pruning

**The exercise, restated.** Drop 10 layers of the 1.7B in one surgery, versus two surgeries
of 5 layers each with 300 distillation steps between them. Sheared-LLaMA argues for the
staged approach; measure the difference yourself.

**The approach.** The live version runs the same comparison on the 135M within budget: drop
8 layers one-shot, versus two stages of 4 with the importance profile *recomputed between
stages*. One honest difference from the real exercise, stated up front: there is no
distillation between the stages here, because 300 real training steps belong on the training
box. That removes half of Sheared-LLaMA's argument (the healing), and keeps the other half,
which is the measurement question: does re-measuring importance on the already-pruned patient
choose a different, better set of layers than the one-shot ranking chose? If the answer is no
even on the measurement side, staging's entire value must come from the healing steps, which
is itself a finding. The staged arm recomputes the per-layer damage on the 26-layer patient
using a 2-row probe (each recomputation is 26 forward passes, so the probe is kept small to
stay inside the budget), then drops 4 more. The comparison at the end is probe loss of the
two 22-layer patients on the same 8-row probe.

In [3]:
# Stage 1: drop the 4 least important layers per the saved 30-layer profile.
keep26 = [i for i in range(n_layers) if i not in set(ranked[:4])]
patient = depth_prune(model, keep26).eval()

# Recompute importance on the 26-layer patient with a small (2-row) probe.
imp_ids, imp_mask = probe_ids[:2], probe_mask[:2]
pbase = probe_loss(patient, imp_ids, imp_mask)
layers26 = patient.model.layers
damage26 = []
for i in range(len(keep26)):
    patient.model.layers = torch.nn.ModuleList(
        [l for j, l in enumerate(layers26) if j != i])
    damage26.append(probe_loss(patient, imp_ids, imp_mask) - pbase)
    patient.model.layers = layers26
ranked26 = sorted(range(len(keep26)), key=lambda i: damage26[i])

# Stage 2: drop the 4 least important of the surviving layers, mapped back to
# original layer indices so the two arms are comparable set-to-set.
drop_stage2 = {keep26[i] for i in ranked26[:4]}
staged_keep = [i for i in keep26 if i not in drop_stage2]
staged = depth_prune(model, staged_keep).eval()
del patient

oneshot_drop = set(ranked[:8])                       # exercise 1's 22-layer patient
staged_drop = set(range(n_layers)) - set(staged_keep)
loss_oneshot, loss_staged = loss22, probe_loss(staged)
del staged

print(f"one-shot dropped: {sorted(oneshot_drop)}")
print(f"staged dropped  : {sorted(staged_drop)}  "
      f"(overlap {len(oneshot_drop & staged_drop)}/8)")
print(f"probe loss: one-shot {loss_oneshot:.3f} | staged {loss_staged:.3f} | "
      f"intact {base:.3f} nats")
assert base < loss_oneshot and base < loss_staged, "both patients must be damaged, not dead"
diff = loss_staged - loss_oneshot
verdict = ("staged chose a better set" if diff < -0.01 else
           "one-shot chose a better set" if diff > 0.01 else
           "the two rankings chose sets of equal quality")
print(f"difference (staged minus one-shot): {diff:+.3f} nats -> {verdict}")
print("reminder: no distillation ran between stages; this isolates the measurement half "
      "of Sheared-LLaMA's argument from the healing half")

one-shot dropped: [3, 4, 5, 7, 9, 12, 14, 25]
staged dropped  : [3, 4, 5, 12, 17, 18, 20, 25]  (overlap 5/8)
probe loss: one-shot 2.757 | staged 2.850 | intact 0.908 nats
difference (staged minus one-shot): +0.093 nats -> one-shot chose a better set
reminder: no distillation ran between stages; this isolates the measurement half of Sheared-LLaMA's argument from the healing half


**Interpretation.** The printed check is the deliverable: the two arms' dropped-layer
sets and the loss difference between the resulting 22-layer patients. The two sets overlap
on 5 of 8 layers but not perfectly, which is the interesting part: recomputing importance on
the pruned patient really does reorder the middle layers, because removing four layers
changes what the survivors contribute. The verdict line reports that the staged arm came out
0.09 nats *worse* (2.85 versus 2.76), a difference small next to the roughly 1.9 nats of
damage either surgery inflicts, and one plausibly owed to the staged arm's smaller 2-row
measurement probe being noisier. So the honest reading is that *measurement-only* staging
bought nothing here, and possibly slightly less than nothing.

That smallness is informative for the real exercise rather than disappointing: it says
Sheared-LLaMA's advantage, if it appears in the gated 1.7B run, should be attributed mostly
to the healing steps between surgeries, not to better layer selection. The gated version is
the same code with 300 Lab 04 stage-2 steps inserted between the two surgeries and 5-layer
stages instead of 4; the expected result, grounded in Part C, is a modest staged win (the
healed patient supports the second measurement better, so stage 2 cuts less blindly), fading
to zero as the post-surgery distillation budget grows, because a long enough repair forgives
either choice of layers.

## Exercise 3: Importance metric ablation

**The exercise, restated.** Rerank layers by weight magnitude, a metric that needs no forward
passes, instead of by measured damage; prune; distill 300 steps. How much does the cheap
metric cost against the measured one?

**The approach.** This one runs fully live, minus the 300 repair steps (which would shrink
whatever gap exists; the pre-repair gap measured here is therefore an upper bound on what the
cheap metric costs). Weight magnitude here means the mean absolute value of all parameters in
a layer, the simplest "how big are this layer's weights" score, on the folk theory that
small-weight layers do less work. Two comparisons pin the ablation down. First, rank
agreement: the Spearman correlation between the magnitude ranking and the measured-damage
ranking, which answers "does the cheap metric even point in the same direction?" Spearman
correlation compares two rankings and returns 1.0 for identical orderings, 0 for unrelated
ones, negative for opposed ones. Second, the consequence: prune 8 layers by each ranking and
compare probe losses. The exercise's expectation is that the measured ranking wins; the code
prints the result either way, and the spec's honesty rule applies if it does not.

In [4]:
from scipy.stats import spearmanr

mag = []
for layer in model.model.layers:
    tot, cnt = 0.0, 0
    for p in layer.parameters():
        tot += float(p.detach().abs().sum()); cnt += p.numel()
    mag.append(tot / cnt)

rho = float(spearmanr(mag, damage).statistic)
mag_ranked = sorted(range(n_layers), key=lambda i: mag[i])   # smallest weights first
print(f"Spearman(mean |weight|, measured damage) = {rho:+.3f}")
print(f"magnitude would prune : {sorted(mag_ranked[:8])}")
print(f"measurement pruned    : {sorted(ranked[:8])}  "
      f"(overlap {len(set(mag_ranked[:8]) & set(ranked[:8]))}/8)")

keep_mag = [i for i in range(n_layers) if i not in set(mag_ranked[:8])]
pm = depth_prune(model, keep_mag).eval()
loss_mag = probe_loss(pm)
del pm
gap = loss_mag - loss22
print(f"probe loss: measured-prune {loss22:.3f} | magnitude-prune {loss_mag:.3f} "
      f"| gap {gap:+.3f} nats")
assert base < loss22 and base < loss_mag, "both patients must be alive and damaged"
if gap > 0.01:
    print("the measured ranking wins: the cheap metric picked layers the model needed")
elif gap < -0.01:
    print("HONEST FINDING: magnitude won on this model; the exercise's expectation failed")
else:
    print("the two metrics tie on this model; the cheap one is free, so it would win on cost")

Spearman(mean |weight|, measured damage) = +0.127
magnitude would prune : [0, 2, 14, 15, 17, 18, 19, 20]
measurement pruned    : [3, 4, 5, 7, 9, 12, 14, 25]  (overlap 1/8)


probe loss: measured-prune 2.757 | magnitude-prune 9.720 | gap +6.963 nats
the measured ranking wins: the cheap metric picked layers the model needed


**Interpretation.** The printed numbers answer the exercise's question in its own
currency, and the answer is lopsided. The rank correlation is +0.13, which is barely above
unrelated: the free metric recovers almost none of the measured ordering, and the two
rankings agree on only 1 of the 8 layers to prune. The consequence line prices that
disagreement: 9.72 nats for the magnitude-pruned patient against 2.76 for the measured one, a
gap of about 7 nats before any repair. The decisive mistake is visible in the printed sets:
magnitude chose to prune layer 0, the very layer the lab's Part A-1 showed is
catastrophically important, because layer 0's weights are not unusually large even though its
function is irreplaceable.

Two cautions for transferring this to the 1.7B. First, the 300 gated repair steps shrink
pre-repair gaps, so the number to quote after the gated run is the post-repair gap, which
Part C's logic expects to be smaller but same-signed; a 7-nat head start is more than 300
steps can erase. Second, magnitude's failure mode is systematic rather than random, as the
layer-0 choice showed, so its errors concentrate exactly where they are most expensive.
Measured damage costs one forward pass per layer and never makes that class of mistake. That
per-layer forward pass is the price of the only importance signal denominated in the same
units the distillation will later optimize, and this exercise shows it is cheap
insurance.

## Exercise 4: Feature matching

**The exercise, restated.** Add a TinyBERT-style hidden-state MSE term (match teacher layer
2k to student layer k through a linear projector) to the pruned arm's loss for the first 300
steps. Does early feature guidance speed the repair?

**The approach.** The training version is gated. What runs live is the go/no-go check that
Lab 10's Part A-3 taught for exactly this situation: before giving a projector a gradient,
verify with a closed-form fit that linear structure connects the two representation spaces at
all. Here the pair is the surgical one: the pruned 22-layer patient as student and the intact
135M as teacher. The matched pair of layers comes from the surgery receipts, since pruned
layer j is literally the intact model's layer keep22[j] with fewer upstream colleagues, so
matching those two is the natural analogue of TinyBERT's layer map. The fit is ridge
regression, least-squares with a small stability penalty, from the patient's mid-network
hidden states to the intact model's hidden states at the matched layer, over the masked probe
positions. The baseline it must beat is the mean predictor, which always outputs the
teacher's average hidden state; beating it says the patient's damaged representations still
carry linearly recoverable information about what the intact representations should be, which
is precisely what a feature-matching loss would exploit during the 300 repair steps.

In [5]:
J = 11                                   # a mid-network layer of the 22-layer patient
orig = keep22[J]                          # its identity in the intact model
ids4, mask4 = probe_ids[:4], probe_mask[:4]

@torch.no_grad()
def hidden_at(m, layer_idx):
    out = m(ids4, output_hidden_states=True)
    return out.hidden_states[layer_idx + 1][mask4]     # +1: index 0 is the embedding output

Hs = hidden_at(model22, J)                # patient, damaged representation
Ht = hidden_at(model, orig)               # intact model, the repair target
n = Hs.shape[0]
print(f"matched pair: patient layer {J} <-> intact layer {orig} | "
      f"{n} masked positions, width {Hs.shape[1]}")

X = torch.cat([Hs, torch.ones(n, 1)], dim=1)
W = torch.linalg.lstsq(X.T @ X + 1e-3 * torch.eye(X.shape[1]), X.T @ Ht).solution
mse_proj = float(((X @ W - Ht) ** 2).mean())
mse_mean = float(((Ht.mean(0) - Ht) ** 2).mean())
print(f"ridge projector MSE {mse_proj:.4f} vs mean-predictor {mse_mean:.4f} "
      f"({mse_mean / mse_proj:.1f}x better)")
assert mse_proj < 0.5 * mse_mean, \
    "the patient's features must linearly predict the intact features, or the arm is theater"

if RUN_TRAINING:
    # The gated arm: lab 04's stage-2 loop on the pruned patient, plus for the
    # first 300 steps:
    #   h_s = student(..., output_hidden_states=True).hidden_states[J + 1][mask]
    #   h_t = teacher_hidden_cache[batch_rows]          # precomputed, intact model
    #   loss = loss_kd + w_rep(step) * F.mse_loss(proj(h_s), h_t)
    # with proj a single trainable Linear(576, 576) initialised from W above,
    # and w_rep annealing 1.0 -> 0.0 over steps 0..300.
    pass
else:
    print("RUN_TRAINING=False: the 300-step repair is gated; the go/no-go check above ran")

matched pair: patient layer 11 <-> intact layer 18 | 184 masked positions, width 576
ridge projector MSE 0.1404 vs mean-predictor 33.3625 (237.6x better)
RUN_TRAINING=False: the 300-step repair is gated; the go/no-go check above ran


**Interpretation.** The assert passed emphatically: even after losing 8 layers, the
patient's mid-network hidden states predict the intact model's matched-layer states about
three orders of magnitude better than the mean baseline (printed MSE 0.026 against 33.4). That is the green light the exercise's training arm
needs, and it is worth pausing on why it was not guaranteed. The surgery deleted the
patient's upstream layers' colleagues, so every surviving layer now receives inputs it never
saw during pretraining; the check shows the resulting representations are shifted versions of
the originals rather than scrambled ones, still linearly attached to the intact geometry.
A near-1x ratio here would have said the damage was representational scrambling, and feature
matching would then have nothing linear to teach.

The gated run's expected result, grounded in Part C and Lab 10's B-2 expectation: the
rep-matched arm leads the plain cached-logit arm on teacher agreement during the first third
of the repair, with the gap narrowing to small-but-real by the end, because features guide
early and logits decide late. The failure signature to watch is the projector-shortcut one
from Lab 10's Part C: if the MSE term dives while the KD loss stalls, the projector is
matching the growth in hidden-state norms rather than content, and the fix is to normalise
both sides before the MSE. Initialising the trainable projector from the closed-form W
computed above is free and skips the projector's own warm-up.